PSPPrograms (1) ───┬──→ (M) PSPRequiredTests ──→ (M) PSPTestResults
│
├──→ (M) PSPRequiredFollowUps ──→ (M) PSPFollowUpRecords
│
├──→ (M) PSPRequiredDataEntries
│
├──→ (M) PSPAdverseEvents
│
├──→ (M) DoctorNotes
│
└──→ (M) PSPParticipations ──→ (M) PSPPatients ──┬──→ (M) PSPeRXs ──→ (M) PSPDispensations
│
├──→ (M) PSPTestResults
│
├──→ (M) PSPFollowUpRecords
│
├──→ (M) PSPAdverseEvents
│
└──→ (M) DoctorNotes


---

## القسم الرابع: ملخص العدادات

| القسم | عدد الجداول |
|-------|--------------|
| الجداول القديمة | 10 |
| الجداول الجديدة | 7 |
| **الإجمالي** | **17** |


### 🤔 رأيي في الجداول الجديدة وإمكانية الدمج

تحليلي للجداول الـ 7 الجديدة:
الجدول	النوع	يمكن دمجه مع؟	رأيي
PSPRequiredTests	تكوين (Configuration)	❌ لا	يبقى منفصل - منطق مختلف
PSPRequiredFollowUps	تكوين (Configuration)	❌ لا	يبقى منفصل - منطق مختلف
PSPRequiredDataEntries	تكوين (Configuration)	❌ لا	يبقى منفصل - منطق مختلف
PSPTestResults	تنفيذ (Execution)	✅ مع PSPRequiredTests؟	لا - تكوين vs تنفيذ
PSPFollowUpRecords	تنفيذ (Execution)	✅ مع PSPRequiredFollowUps؟	لا - تكوين vs تنفيذ
PSPAdverseEvents	تنفيذ (Execution)	❌ لا	يبقى منفصل - مستقل بذاته
DoctorNotes	تنفيذ (Execution)	✅ مع PSPFollowUpRecords؟	ممكن جزئياً
اقتراحي للدمج (إن أردت تقليل عدد الجداول):
الخيار 1: دمج DoctorNotes مع PSPFollowUpRecords
لماذا؟

كلاهما تسجيلات من الطبيب

كلاهما مرتبط بمريض وبرنامج

يمكن جعل الملاحظات حقلاً إضافياً في متابعة

الهيكل المقترح بعد الدمج:

markdown
### PSPFollowUpRecords (مدمج)
- الأعمدة الأصلية كلها موجودة
- + Notes (nvarchar(2000)) - ملاحظة أساسية
- + DoctorNotesJSON (nvarchar(max)) - للملاحظات المتعددة إن احتاجها
توفير: جدول واحد (من 17 إلى 16 جدولاً)

الخيار 2: توحيد الجداول الثلاثة للتكوين (Configuration)
دمج PSPRequiredTests + PSPRequiredFollowUps + PSPRequiredDataEntries في جدول واحد:

الجدول الموحد: PSPRequiredItems

العمود	الوصف
RequiredItemID	PK
ProgramID	FK
ItemType	1=تحليل, 2=متابعة, 3=بيانات
Title	اسم العنصر
Description	وصف
RequiredFrom	1=مريض, 2=طبيب, 3=كلاهما
RequiredTiming	1=قبل, 2=أثناء, 3=بعد, 4=دوري
FrequencyDays	للتكرار
ConfigJSON	nvarchar(max) - للتخزين المرن
محتوى ConfigJSON حسب ItemType:

تحليل: { "NormalRangeMin": 0, "NormalRangeMax": 100, "Unit": "mg/dL" }

متابعة: { "FollowUpType": 1, "DurationMinutes": 30, "ReminderDaysBefore": 3 }

بيانات: { "DataEntryType": 3, "AllowedValues": ["خيار1","خيار2"] }

توفير: جدولان (من 17 إلى 15 جدولاً)

الخيار 3: توحيد الجداول الثلاثة للتنفيذ (Execution)
دمج PSPTestResults + PSPFollowUpRecords في جدول واحد:

الجدول الموحد: PSPExecutionRecords

العمود	الوصف
ExecutionRecordID	PK
PatientID	FK
PerformedByUserID	FK
RecordType	1=نتيجة تحليل, 2=سجل متابعة
RequiredItemID	FK (لجدول PSPRequiredItems الموحد)
PerformedDate	datetime
ResultJSON	nvarchar(max)
توفير: جدول واحد (من 17 إلى 14 جدولاً)

📊 مقارنة الخيارات:
الخيار	عدد الجداول	المرونة	سهولة الاستعلام	تعقيد التطوير
الوضع الحالي	17 جدولاً	عالية جداً	سهلة	متوسط
الخيار 1 (دمج DoctorNotes)	16 جدولاً	عالية	سهلة	منخفض
الخيار 2 (توحيد التكوين)	15 جدولاً	متوسطة	متوسطة	متوسط
الخيار 3 (توحيد التنفيذ)	14 جدولاً	متوسطة	متوسطة	مرتفع
💭 رأيي النهائي (الأهم)
هل الوضع الحالي صحيح؟
نعم، الوضع الحالي صحيح من الناحية المعمارية.

الأسباب:

فصل التكوين عن التنفيذ - مبدأ أساسي في قواعد البيانات

كل جدول له غرض محدد - يسهل الصيانة والتطوير

مرونة عالية - يمكن إضافة أنواع جديدة بدون تعديل هيكل الجداول

استعلامات بسيطة - كل جدول لوحده، بدون الحاجة لـ JSON parsing

هل تحتاج للدمج؟
لا، لا أنصح بالدمج في حالتك.

الأسباب:

المشروع لا يزال في بدايته - 17 جدولاً عدد مقبول جداً لنظام PSP متكامل

الدمج قد يسبب تعقيدات لاحقة - صيانة JSON صعبة، استعلامات أبطأ

الوضوح أهم من تقليل عدد الجداول - فريقك سيفهم النظام بسهولة أكبر

يمكنك إضافة الجداول تدريجياً - ابدأ بالأساسي، زود لاحقاً

[#007]

شادي، فكرة ممتازة ✅

تقسيم المهام **بعدد الساعات** بدلاً من الأيام يعطي صورة أوضح عن **الحجم الحقيقي** للعمل ويساعد في:

1. **تخطيط أفضل** (كل فريق يعرف كم ساعة سيحتاج)
2. **تتبع الأداء** (هل نحن أسرع أم أبطأ من المتوقع)
3. **التحسين المستمر** (بعد كل إصدار، نعرف أين ضاعت الساعات)

---

## 📊 تقسيم المهام بالساعات

### المرحلة 0: التأسيس (إجمالي: 35-45 ساعة)

| # | المهمة | الساعات | المسؤول | الوصف |
|---|--------|---------|---------|-------|
| 0.1 | إنشاء RCL للمكونات المشتركة | 3-4 | شادي | RubikCare.Shared.UI + تكوين المشروع |
| 0.2 | تجربة Blazor Hybrid (صفحة تجريبية) | 4-5 | شادي | اختبار الجدوى وحل المشاكل الأولى |
| 0.3 | إعداد تشفير البيانات | 6-8 | شادي | تشفير الحساسة في DB + Service Layer |
| 0.4 | توثيق هيكل قاعدة البيانات الجديد | 3-4 | يوسف | كتابة الـ Specifications للجداول الجديدة |
| 0.5 | إنشاء الـ Migration للجداول الجديدة | 3-4 | يوسف | Code-First + تحديث DbContext |
| 0.6 | إعداد Firebase (Messaging + Storage) | 5-6 | شادي | حساب + تهيئة + Service للتكامل |
| 0.7 | صفحة السياسات (Terms + Privacy) | 4-5 | يوسف | Blazor Hybrid + محتوى ثابت |
| 0.8 | مراجعة واختبار المرحلة | 4-5 | شادي + يوسف | ضمان عمل كل شيء معاً |

---

### المرحلة 1: الإصدار التجريبي (Beta) - إجمالي: 80-100 ساعة

#### المجموعة 1: الإشعارات وقائمة المرضى (30-35 ساعة)

| # | المهمة | الساعات | المسؤول | الوصف |
|---|--------|---------|---------|-------|
| 1.1 | تفعيل الإشعارات الأساسية (FCM) | 8-10 | شادي | تسجيل جهاز + إرسال إشعارات بسيطة |
| 1.2 | واجهة إدارة الإشعارات في التطبيق | 4-5 | يوسف | عرض الإشعارات الواردة |
| 1.3 | **قائمة مرضى الطبيب** (قراءة فقط) | 8-10 | شادي | API + واجهة Blazor |
| 1.4 | تفاصيل مريض للطبيب (متابعة بسيطة) | 6-7 | شادي | عرض تاريخ البرامج والصرف |
| 1.5 | ربط الإشعارات مع قائمة المرضى | 3-4 | شادي | إشعار عند صرف المريض |

#### المجموعة 2: المريض والصيدلي (30-35 ساعة)

| # | المهمة | الساعات | المسؤول | الوصف |
|---|--------|---------|---------|-------|
| 1.6 | **سجل صحي بسيط للمريض** (قراءة فقط) | 6-8 | يوسف | عرض تاريخ البرامج والصرف |
| 1.7 | تفاصيل البرنامج للمريض (مطورة) | 5-6 | يوسف | عرض التقدم والالتزام |
| 1.8 | تأكيد صرف + سجل للصيدلي | 6-8 | يوسف | API + واجهة Blazor |
| 1.9 | سجل صرفيات الصيدلي (بسيط) | 5-6 | يوسف | عرض تاريخ الصرف |
| 1.10 | ربط الصيدلي مع الإشعارات | 4-5 | يوسف | إشعار عند طلب صرف جديد |

#### المجموعة 3: Pharma Dashboard والاختبارات (20-30 ساعة)

| # | المهمة | الساعات | المسؤول | الوصف |
|---|--------|---------|---------|-------|
| 1.11 | Dashboard مبسط لشركة الأدوية | 8-10 | شادي | إحصائيات أساسية (أعداد، صرفيات) |
| 1.12 | تكامل البيانات بين المكونات | 4-5 | شادي | ضمان تدفق البيانات الصحيح |
| 1.13 | اختبار الدورة الكاملة | 6-8 | شادي + يوسف | طبيب ← مريض ← صيدلي |
| 1.14 | إصلاح مشاكل Beta الحرجة | 4-6 | شادي + يوسف | حسب نتائج الاختبار |

---

### المرحلة 2: الإصدار الأول الكامل (v1.0) - إجمالي: 100-130 ساعة

#### المجموعة 1: تسجيلات الطبيب والمريض (35-45 ساعة)

| # | المهمة | الساعات | المسؤول | الوصف |
|---|--------|---------|---------|-------|
| 2.1 | **تسجيل ملاحظات الطبيب على المريض** | 10-12 | شادي | API + واجهة + تخزين |
| 2.2 | **تسجيل المريض للآثار الجانبية** | 8-10 | يوسف | واجهة + API + تحليل بسيط |
| 2.3 | تصميم نموذج بيانات قابل للتحليل | 5-6 | شادي | هيكل يسمح بالتقارير |
| 2.4 | ربط التسجيلات مع برنامج الدعم | 6-8 | شادي | لكل برنامج تسجيلاته المطلوبة |
| 2.5 | واجهة عرض التسجيلات للطبيب | 5-8 | يوسف | عرض ملاحظات المريض |

#### المجموعة 2: برامج الدعم المتقدمة (30-35 ساعة)

| # | المهمة | الساعات | المسؤول | الوصف |
|---|--------|---------|---------|-------|
| 2.6 | **تصميم برنامج دعم متقدم** | 8-10 | شادي | إضافة تحاليل وتسجيلات مطلوبة |
| 2.7 | واجهة إنشاء برنامج متقدم (Pharma) | 6-8 | شادي | Web Dashboard |
| 2.8 | تنفيذ التحاليل المطلوبة في التطبيق | 6-8 | يوسف | عرض ورفع نتائج |
| 2.9 | تنفيذ مواعيد المتابعة | 5-7 | يوسف | إشعارات + تسجيل |

#### المجموعة 3: الصيدلي المتقدم (20-25 ساعة)

| # | المهمة | الساعات | المسؤول | الوصف |
|---|--------|---------|---------|-------|
| 2.10 | **طلب أصناف من الصيدلي** | 6-8 | يوسف | API + واجهة + إشعار للشركة |
| 2.11 | **المطالبة بالتعويضات** | 6-8 | يوسف | إنشاء طلب تعويض + حالة |
| 2.12 | لوحة تحكم الصيدلي (مطورة) | 4-5 | يوسف | طلبات + تعويضات + صرفيات |
| 2.13 | ربط طلبات الصيدلي مع الشركة | 3-4 | شادي | إشعار + Dashboard الشركة |

#### المجموعة 4: التقارير والترجمة (15-25 ساعة)

| # | المهمة | الساعات | المسؤول | الوصف |
|---|--------|---------|---------|-------|
| 2.14 | **لوحات تحكم متقدمة** | 8-10 | شادي | لكل مستخدم (Pharma, Doctor, Rep) |
| 2.15 | تقارير تحليلية (آثار جانبية، نتائج) | 5-8 | شادي | مستخرجة من التسجيلات |
| 2.16 | **ترجمة الهاتف بالكامل** | 6-8 | يوسف | عربي/إنجليزي (RTL/LTR) |

---

### المرحلة 3: ما بعد الإصدار (Post v1.0) - إجمالي: 60-80 ساعة

| # | المهمة | الساعات | المسؤول | الأولوية |
|---|--------|---------|---------|----------|
| 3.1 | Unit tests (API والمكتبات الأساسية) | 15-20 | شادي | عالية |
| 3.2 | **النسخ الاحتياطي للمستخدم** | 12-15 | يوسف | عالية |
| 3.3 | اختبارات آلية شاملة | 10-12 | شادي + يوسف | متوسطة |
| 3.4 | دفع أون لاين | 12-15 | شادي | متوسطة |
| 3.5 | ربط مع شركة شحن | 8-10 | يوسف | منخفضة |
| 3.6 | تحسينات إضافية حسب feedback | 8-10 | شادي + يوسف | حسب الحاجة |

---

## 📊 ملخص إجمالي الساعات

| المرحلة | الحد الأدنى | الحد الأقصى | المتوقع |
|---------|-------------|-------------|---------|
| المرحلة 0 (التأسيس) | 35 | 45 | 40 |
| المرحلة 1 (Beta) | 80 | 100 | 90 |
| المرحلة 2 (v1.0) | 100 | 130 | 115 |
| المرحلة 3 (Post) | 60 | 80 | 70 |
| **الإجمالي** | **275** | **355** | **315** |

**بمعدل 40 ساعة أسبوعياً (لكل منكما):**
- 315 ساعة إجمالي / 2 شخص = 157.5 ساعة لكل شخص
- 157.5 / 40 = **4 أسابيع تقريباً** (لشخصين)

**ملاحظة:** هذا تقدير متفائل. الأفضل إضافة 20% احتياطي = **5 أسابيع**

---

## 🎯 جدول زمني بالأيام (بناءً على 8 ساعات عمل يومياً)

| المرحلة | الساعات | أيام (شخص واحد) | أيام (شخصين) |
|---------|---------|-----------------|---------------|
| المرحلة 0 | 40 | 5 أيام | **2.5 يوم** |
| المرحلة 1 | 90 | 11.5 يوم | **5.5 يوم** |
| المرحلة 2 | 115 | 14.5 يوم | **7 يوم** |
| المرحلة 3 | 70 | 9 يوم | **4.5 يوم** |
| **الإجمالي** | **315** | **40 يوم** | **20 يوم** |

**مع الاحتياطي (20%):** 24 يوم عمل = **حوالي 5 أسابيع** (لشخصين)

---

## 📋 صيغة Jira (جاهزة للاستيراد)

### تنسيق CSV للاستيراد إلى Jira

```csv
Summary,Description,Assignee,Story Points,Priority,Epic Link
0.1 إنشاء RCL للمكونات المشتركة,إنشاء RubikCare.Shared.UI + تكوين المشروع,شادي,4,High,التأسيس
0.2 تجربة Blazor Hybrid,اختبار الجدوى وحل المشاكل الأولى,شادي,5,High,التأسيس
0.3 إعداد تشفير البيانات,تشفير البيانات الحساسة في DB + Service Layer,شادي,7,High,التأسيس
0.4 توثيق هيكل قاعدة البيانات الجديد,كتابة الـ Specifications للجداول الجديدة,يوسف,4,Medium,التأسيس
0.5 إنشاء الـ Migration للجداول الجديدة,Code-First + تحديث DbContext,يوسف,4,High,التأسيس
0.6 إعداد Firebase,حساب + تهيئة + Service للتكامل,شادي,6,High,التأسيس
0.7 صفحة السياسات,Blazor Hybrid + محتوى ثابت,يوسف,5,Medium,التأسيس
0.8 مراجعة واختبار المرحلة,ضمان عمل كل شيء معاً,شادي+يوسف,5,High,التأسيس
1.1 تفعيل الإشعارات الأساسية (FCM),تسجيل جهاز + إرسال إشعارات بسيطة,شادي,9,High,Beta
1.2 واجهة إدارة الإشعارات,عرض الإشعارات الواردة,يوسف,5,Medium,Beta
1.3 قائمة مرضى الطبيب,API + واجهة Blazor,شادي,9,High,Beta
1.4 تفاصيل مريض للطبيب,عرض تاريخ البرامج والصرف,شادي,7,High,Beta
1.5 ربط الإشعارات مع قائمة المرضى,إشعار عند صرف المريض,شادي,4,Medium,Beta
1.6 سجل صحي بسيط للمريض,عرض تاريخ البرامج والصرف,يوسف,7,High,Beta
1.7 تفاصيل البرنامج للمريض مطورة,عرض التقدم والالتزام,يوسف,6,High,Beta
1.8 تأكيد صرف + سجل للصيدلي,API + واجهة Blazor,يوسف,7,High,Beta
1.9 سجل صرفيات الصيدلي بسيط,عرض تاريخ الصرف,يوسف,6,Medium,Beta
1.10 ربط الصيدلي مع الإشعارات,إشعار عند طلب صرف جديد,يوسف,5,Medium,Beta
1.11 Dashboard مبسط لشركة الأدوية,إحصائيات أساسية,شادي,9,High,Beta
1.12 تكامل البيانات بين المكونات,ضمان تدفق البيانات الصحيح,شادي,5,High,Beta
1.13 اختبار الدورة الكاملة,طبيب ← مريض ← صيدلي,شادي+يوسف,7,High,Beta
1.14 إصلاح مشاكل Beta الحرجة,حسب نتائج الاختبار,شادي+يوسف,6,High,Beta
2.1 تسجيل ملاحظات الطبيب على المريض,API + واجهة + تخزين,شادي,11,High,v1.0
2.2 تسجيل المريض للآثار الجانبية,واجهة + API + تحليل بسيط,يوسف,9,High,v1.0
2.3 تصميم نموذج بيانات قابل للتحليل,هيكل يسمح بالتقارير,شادي,6,High,v1.0
2.4 ربط التسجيلات مع برنامج الدعم,لكل برنامج تسجيلاته المطلوبة,شادي,7,High,v1.0
2.5 واجهة عرض التسجيلات للطبيب,عرض ملاحظات المريض,يوسف,7,Medium,v1.0
2.6 تصميم برنامج دعم متقدم,إضافة تحاليل وتسجيلات مطلوبة,شادي,9,High,v1.0
2.7 واجهة إنشاء برنامج متقدم,Web Dashboard,شادي,7,High,v1.0
2.8 تنفيذ التحاليل المطلوبة في التطبيق,عرض ورفع نتائج,يوسف,7,High,v1.0
2.9 تنفيذ مواعيد المتابعة,إشعارات + تسجيل,يوسف,6,Medium,v1.0
2.10 طلب أصناف من الصيدلي,API + واجهة + إشعار للشركة,يوسف,7,High,v1.0
2.11 المطالبة بالتعويضات,إنشاء طلب تعويض + حالة,يوسف,7,High,v1.0
2.12 لوحة تحكم الصيدلي مطورة,طلبات + تعويضات + صرفيات,يوسف,5,Medium,v1.0
2.13 ربط طلبات الصيدلي مع الشركة,إشعار + Dashboard الشركة,شادي,4,High,v1.0
2.14 لوحات تحكم متقدمة,لكل مستخدم,شادي,9,High,v1.0
2.15 تقارير تحليلية,آثار جانبية، نتائج,شادي,7,High,v1.0
2.16 ترجمة الهاتف بالكامل,عربي/إنجليزي (RTL/LTR),يوسف,7,Medium,v1.0
3.1 Unit tests,API والمكتبات الأساسية,شادي,18,High,Post-v1.0
3.2 النسخ الاحتياطي للمستخدم,مساحة سحابية خاصة,يوسف,14,High,Post-v1.0
3.3 اختبارات آلية شاملة,Manual + Automated,شادي+يوسف,11,Medium,Post-v1.0
3.4 دفع أون لاين,Integration مع بوابة دفع,شادي,14,Medium,Post-v1.0
3.5 ربط مع شركة شحن,Integration,يوسف,9,Low,Post-v1.0
3.6 تحسينات إضافية حسب feedback,تطوير مستمر,شادي+يوسف,9,Low,Post-v1.0
```

---

## 📝 ملاحظات لاستيراد Jira

1. **Story Points:** حسبت بنظام Fibonacci (1,2,3,5,7,9,11,14,18) حيث كل نقطة = ساعة واحدة تقريباً
2. **الأولويات:**
   - High → يجب أن تكتمل قبل الإصدار
   - Medium → مهمة ولكن يمكن تأجيلها للإصدار التالي إذا لزم الأمر
   - Low → تحسينات
3. **Epic Link:** يمكنك إنشاء 4 Epics:
   - `التأسيس` (Foundation)
   - `Beta` (Beta Release)
   - `v1.0` (First Full Release)
   - `Post-v1.0` (Future Enhancements)

---

## 🚀 الخطوة التالية

بعد موافقتك على:
1. تقدير الساعات
2. تقسيم المهام
3. صيغة Jira

**هل نبدأ في تنفيذ المهمة الأولى (0.1: إنشاء RCL للمكونات المشتركة)؟**

أم تريد تعديل أي شيء في التقديرات أو التوزيع؟

---

بانتظار ردك [#008] 🚀